# RAC Vision: detector de EPP unificado (un solo pase)

Hoy RAC corre **dos** YOLOv8n por cuadro: cabezas con/sin casco (`detector_casco.onnx`) y personas/chalecos
(`detector_chaleco.onnx`), unos 48 ms + 47 ms en el servidor. Este notebook entrena **uno solo** con todas las
clases: el preprocesado y la inferencia se hacen una vez y las cajas salen del mismo cuadro.

Clases: `0 persona`, `1 casco`, `2 sin_casco`, `3 chaleco`, `4 antiparras`, `5 sin_antiparras`, `6 guantes`, `7 sin_guantes`.
Antiparras y guantes son nuevas: suman valor en minería y obra pesada.

Datos:
- [Construction-PPE](https://docs.ultralytics.com/datasets/detect/construction-ppe) (1416 fotos, AGPL-3.0): personas, casco, chaleco, antiparras, guantes y sus "sin".
- [Hard Hat Detection](https://huggingface.co/datasets/dgural/hard-hat-detection) (5000 fotos, CC0): más cabezas con y sin casco desde muchos ángulos.
- Opcional: tus fotos (`fotos_rac.zip` con `con_casco/` y `sin_casco/`), sobre todo **pelo oscuro, gorras y capuchas**
  sin casco: es el error que hoy tiene el modelo.

Aumentos para mina y obra: niebla y polvo, destellos de linternas, sombras duras, luz de sodio (todo naranja),
poca luz, desenfoque de movimiento y ruido de sensor.

*Entorno de ejecución → Cambiar tipo → GPU T4*, y *Ejecutar todas*. Al final se descarga `detector_epp.onnx`.
Para usarlo en RAC falta el paso 2 del plan (`docs/plan-vision-industrial.md`): el código todavía espera dos modelos.

## 1. Librerías y GPU

In [ ]:
!nvidia-smi -L
!pip install -q "ultralytics>=8.3,<8.4" huggingface_hub "albumentations>=1.4,<2"

## 2. Construction-PPE → clases unificadas

In [ ]:
import json, random, shutil, zipfile, urllib.request
from collections import Counter
from pathlib import Path

CLASES = ["persona", "casco", "sin_casco", "chaleco", "antiparras", "sin_antiparras", "guantes", "sin_guantes"]
DST = Path("/content/epp_yolo")
for split in ("train", "val", "test"):
    (DST / "images" / split).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / split).mkdir(parents=True, exist_ok=True)
conteo = Counter()

urllib.request.urlretrieve("https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip",
                           "/content/construction-ppe.zip")
zipfile.ZipFile("/content/construction-ppe.zip").extractall("/content/ppe")
SRC = next(p.parent for p in Path("/content/ppe").rglob("images") if (p / "train").exists())
# construction-ppe.yaml: 0 helmet, 1 gloves, 2 vest, 3 boots, 4 goggles, 5 none, 6 Person, 7 no_helmet,
# 8 no_goggle, 9 no_gloves, 10 no_boots. Botas y "none" se descartan.
MAPA_PPE = {6: 0, 0: 1, 7: 2, 2: 3, 4: 4, 8: 5, 1: 6, 9: 7}
for split in ("train", "val", "test"):
    for img in (SRC / "images" / split).glob("*"):
        lbl = SRC / "labels" / split / (img.stem + ".txt")
        lineas = []
        if lbl.exists():
            for l in lbl.read_text().splitlines():
                c, *caja = l.split()
                if int(c) in MAPA_PPE:
                    lineas.append(" ".join([str(MAPA_PPE[int(c)]), *caja]))
                    conteo[CLASES[MAPA_PPE[int(c)]]] += 1
        shutil.copy(img, DST / "images" / split / ("ppe_" + img.name))
        (DST / "labels" / split / ("ppe_" + img.stem + ".txt")).write_text("\n".join(lineas))
print(conteo)

## 3. Hard Hat (más cabezas con y sin casco)

In [ ]:
from huggingface_hub import snapshot_download
HH = Path(snapshot_download("dgural/hard-hat-detection", repo_type="dataset", local_dir="/content/hardhat"))
samples = json.load(open(HH / "samples.json"))["samples"]
random.seed(0)
random.shuffle(samples)
n_val = int(len(samples) * 0.15)
MAPA_HH = {"helmet": 1, "head": 2, "person": 0}
for i, s in enumerate(samples):
    split = "val" if i < n_val else "train"
    img = HH / s["filepath"]
    if not img.exists():
        img = HH / "data" / Path(s["filepath"]).name
    lineas = []
    for d in (s.get("ground_truth") or {}).get("detections", []):
        c = MAPA_HH.get(d["label"].lower())
        if c is None:
            continue
        x, y, w, h = d["bounding_box"]  # FiftyOne: esquina superior izquierda, relativo
        lineas.append(f"{c} {x + w / 2:.6f} {y + h / 2:.6f} {w:.6f} {h:.6f}")
        conteo[CLASES[c]] += 1
    shutil.copy(img, DST / "images" / split / ("hh_" + img.name))
    (DST / "labels" / split / ("hh_" + img.stem + ".txt")).write_text("\n".join(lineas))
print(conteo)

## 4. Aumentos de mina y obra (sobre una copia de train)

Se agrega una versión alterada de cada foto de entrenamiento. Validación y prueba quedan sin tocar para que
la medición sea honesta.

In [ ]:
import albumentations as A
import cv2
import numpy as np


def sodio(im, **kwargs):
    # Luz de sodio / LED monocromática: todo tirando a naranja
    return cv2.addWeighted(im, 0.55, np.full_like(im, (20, 140, 255)), 0.45, 0)


aumentos = A.Compose([
    A.OneOf([A.RandomFog(p=1), A.GaussNoise(p=1), A.ISONoise(p=1)], p=0.6),                  # polvo, niebla, sensor
    A.OneOf([A.RandomSunFlare(src_radius=120, p=1), A.RandomShadow(p=1)], p=0.4),             # linternas, sombras
    A.OneOf([A.Lambda(image=sodio, p=1), A.ToGray(p=1),
             A.RandomBrightnessContrast((-0.6, -0.2), (-0.3, 0.1), p=1)], p=0.6),             # luz pobre
    A.MotionBlur(blur_limit=9, p=0.3),
], bbox_params=A.BboxParams(format="yolo", label_fields=["clases"], min_visibility=0.3))

hechas = 0
for img_path in list((DST / "images" / "train").glob("*")):
    lbl = DST / "labels" / "train" / (img_path.stem + ".txt")
    img = cv2.imread(str(img_path))
    if img is None or not lbl.exists():
        continue
    filas = [l.split() for l in lbl.read_text().splitlines() if l.strip()]
    cajas = [[min(max(float(v), 1e-6), 1.0) for v in f[1:]] for f in filas]
    try:
        r = aumentos(image=img, bboxes=cajas, clases=[int(f[0]) for f in filas])
    except ValueError:  # una caja que sale del cuadro: esa foto no se aumenta
        continue
    cv2.imwrite(str(DST / "images" / "train" / f"aug_{img_path.stem}.jpg"), r["image"])
    (DST / "labels" / "train" / f"aug_{img_path.stem}.txt").write_text(
        "\n".join(f"{c} " + " ".join(f"{v:.6f}" for v in b) for c, b in zip(r["clases"], r["bboxes"])))
    hechas += 1
print("Fotos aumentadas:", hechas)

(DST / "data.yaml").write_text(f"path: {DST}\ntrain: images/train\nval: images/val\ntest: images/test\n"
                               f"names: {dict(enumerate(CLASES))}\n")

## 5. (Opcional) Tus fotos difíciles

Subí `fotos_rac.zip` con `con_casco/` y `sin_casco/`. Sumá todas las que puedas de **pelo oscuro, gorras,
capuchas y cámaras en altura**. Un 30% queda aparte para medir. Si no subís nada (Cancelar), seguí.

In [ ]:
from google.colab import files
fotos_ajuste, fotos_prueba = [], []
subido = files.upload()
if subido:
    zipfile.ZipFile(next(iter(subido))).extractall("/content/rac")
    fotos = [(p, real) for real in ("con_casco", "sin_casco") for p in sorted(Path("/content/rac", real).glob("*.jpg"))]
    random.seed(1)
    random.shuffle(fotos)
    corte = int(len(fotos) * 0.7)
    fotos_ajuste, fotos_prueba = fotos[:corte], fotos[corte:]
print(len(fotos_ajuste), "para ajustar |", len(fotos_prueba), "para probar")

## 6. Entrenar

In [ ]:
from ultralytics import YOLO
modelo = YOLO("yolov8n.pt")
modelo.train(data=str(DST / "data.yaml"), epochs=80, imgsz=640, batch=32, patience=20,
             degrees=15, hsv_v=0.6, fliplr=0.5, mosaic=1.0, close_mosaic=10,
             project="/content/runs", name="epp", exist_ok=True)
metricas = YOLO("/content/runs/epp/weights/best.pt").val(data=str(DST / "data.yaml"), split="test")
for i, nombre in enumerate(CLASES):
    print(f"{nombre:15s} mAP50-95 {metricas.box.maps[i]:.3f}")

## 7. Ajuste con tus fotos (si subiste) y medición

Igual que en `detector_casco_colab.ipynb`: el modelo marca la cabeza y la clase sale de la carpeta.

In [ ]:
def evaluar(ruta, conf=0.35):
    m = YOLO(ruta)
    tabla, ok = Counter(), 0
    for p, real in fotos_prueba:
        clases = {int(c) for c in m.predict(str(p), conf=conf, imgsz=640, verbose=False)[0].boxes.cls}
        pred = "sin_casco" if 2 in clases else "con_casco" if 1 in clases else "nada"
        tabla[(real, pred)] += 1
        ok += pred == real
    print(f"Aciertos: {ok}/{len(fotos_prueba)}", dict(tabla))


ELEGIDO = "epp"
if fotos_ajuste:
    evaluar("/content/runs/epp/weights/best.pt")
    base = YOLO("/content/runs/epp/weights/best.pt")
    for p, real in fotos_ajuste:
        r = base.predict(str(p), conf=0.2, imgsz=640, verbose=False, classes=[1, 2])[0]
        if len(r.boxes) == 0:
            continue
        cx, cy, w, h = r.boxes.xywhn[int(r.boxes.conf.argmax())].tolist()
        clase = 1 if real == "con_casco" else 2
        for k in range(3):  # se repite para que pesen frente a las miles de fotos públicas
            nombre = f"rac_{real}_{p.stem}_{k}"
            shutil.copy(p, DST / "images" / "train" / f"{nombre}.jpg")
            (DST / "labels" / "train" / f"{nombre}.txt").write_text(f"{clase} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    YOLO("/content/runs/epp/weights/best.pt").train(data=str(DST / "data.yaml"), epochs=20, imgsz=640, batch=32,
                                                    lr0=0.002, degrees=15, hsv_v=0.6,
                                                    project="/content/runs", name="epp_ajustado", exist_ok=True)
    evaluar("/content/runs/epp_ajustado/weights/best.pt")
    ELEGIDO = "epp_ajustado"   # si los aciertos bajaron respecto del primero, volvé a "epp"

## 8. Velocidad en CPU (OpenVINO INT8, como en el servidor) y descarga

In [ ]:
import time
final = YOLO(f"/content/runs/{ELEGIDO}/weights/best.pt")
onnx = final.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
ov = final.export(format="openvino", imgsz=640, int8=True, data=str(DST / "data.yaml"))
m = YOLO(ov)
img = str(next((DST / "images" / "test").glob("*")))
m.predict(img, device="cpu", verbose=False)
t = time.perf_counter()
for _ in range(20):
    m.predict(img, device="cpu", verbose=False)
print(f"CPU OpenVINO INT8: {(time.perf_counter() - t) / 20 * 1000:.0f} ms por cuadro (hoy casco + chaleco: unos 95 ms)")
shutil.copy(onnx, "/content/detector_epp.onnx")
files.download("/content/detector_epp.onnx")
files.download(f"/content/runs/{ELEGIDO}/weights/best.pt")